In [7]:
import pandas as pd

# load the raw OpenAQ file
df = pd.read_csv("dubai_openaq_combined_data.csv")

# use the local start date and reduce it to just the date
df["date"] = pd.to_datetime(df["date_local_start"]).dt.date

# keep the main daily value for each parameter
openaq_clean = df.pivot_table(
    index=["location_name", "date"],
    columns="parameter",
    values="value",
    aggfunc="mean"
).reset_index()

# remove pivot formatting
openaq_clean.columns.name = None

# save cleaned file
openaq_clean.to_csv("dubai_openaq_clean_wide.csv", index=False)

print(openaq_clean.head())
print(openaq_clean.columns)
print(openaq_clean.shape)

      location_name        date  humidity  pm10  pm25  temperature
0  Dubai Motor City  2024-10-08      37.3  19.1  18.5         35.0
1  Dubai Motor City  2024-10-09      42.6  28.5  26.9         33.5
2  Dubai Motor City  2024-10-10      45.1  49.8  44.0         33.7
3  Dubai Motor City  2024-10-11      50.9  51.6  43.5         31.1
4  Dubai Motor City  2024-10-14      37.0  28.8  26.6         32.4
Index(['location_name', 'date', 'humidity', 'pm10', 'pm25', 'temperature'], dtype='str')
(1668, 6)


In [10]:
import pandas as pd

openaq = pd.read_csv("dubai_openaq_clean_wide.csv")
weather = pd.read_csv("dubai_meteostat_daily.csv")

openaq["date"] = pd.to_datetime(openaq["date"])
weather["date"] = pd.to_datetime(weather["date"])

# drop weather columns that are fully empty / not useful
weather = weather.drop(columns=[
    "snow_depth_meteostat",
    "wind_gust_meteostat",
    "sunshine_meteostat"
], errors="ignore")

master = openaq.merge(weather, on="date", how="left")
master = master.sort_values(["location_name", "date"]).reset_index(drop=True)

master.to_csv("master_daily.csv", index=False)

print(master.head())
print(master.columns)
print(master.shape)
print(master.isna().sum())

      location_name       date  humidity  pm10  pm25  temperature  \
0  Dubai Motor City 2024-10-08      37.3  19.1  18.5         35.0   
1  Dubai Motor City 2024-10-09      42.6  28.5  26.9         33.5   
2  Dubai Motor City 2024-10-10      45.1  49.8  44.0         33.7   
3  Dubai Motor City 2024-10-11      50.9  51.6  43.5         31.1   
4  Dubai Motor City 2024-10-14      37.0  28.8  26.6         32.4   

   temp_meteostat  temp_min_meteostat  temp_max_meteostat  humidity_meteostat  \
0            33.4                29.0                39.7                  48   
1            32.8                29.9                37.6                  53   
2            32.6                29.5                37.5                  60   
3            32.8                30.0                37.2                  54   
4            33.3                29.5                37.9                  46   

   precipitation_meteostat  wind_speed_meteostat  pressure_meteostat  \
0                      0.0

In [11]:
pm25_df = master[master["pm25"].notna()].copy()
pm10_df = master[master["pm10"].notna()].copy()

pm25_df.to_csv("pm25_analysis.csv", index=False)
pm10_df.to_csv("pm10_analysis.csv", index=False)

print("PM2.5 file shape:", pm25_df.shape)
print("PM10 file shape:", pm10_df.shape)

print("\nRows per location in PM2.5 file:")
print(pm25_df["location_name"].value_counts())

print("\nRows per location in PM10 file:")
print(pm10_df["location_name"].value_counts())

PM2.5 file shape: (1668, 14)
PM10 file shape: (374, 14)

Rows per location in PM2.5 file:
location_name
The Views           606
Serena              535
Dubai Motor City    527
Name: count, dtype: int64

Rows per location in PM10 file:
location_name
The Views           174
Serena              103
Dubai Motor City     97
Name: count, dtype: int64


In [12]:
# convert date to datetime (if not already)
master["date"] = pd.to_datetime(master["date"])

# add time features
master["year"] = master["date"].dt.year
master["month"] = master["date"].dt.month
master["month_name"] = master["date"].dt.month_name()
master["day_of_week"] = master["date"].dt.day_name()
master["is_weekend"] = master["date"].dt.dayofweek >= 5

# simple season function
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

master["season"] = master["month"].apply(get_season)

# save updated master
master.to_csv("master_with_time.csv", index=False)

print(master[["date", "month_name", "season", "day_of_week"]].head())

        date month_name  season day_of_week
0 2024-10-08    October  Autumn     Tuesday
1 2024-10-09    October  Autumn   Wednesday
2 2024-10-10    October  Autumn    Thursday
3 2024-10-11    October  Autumn      Friday
4 2024-10-14    October  Autumn      Monday
